# Deposit Study 2027v16 Lineage Graph

An interactive, layered map of the 2027v16 deposit beta study (`pricing_analytics/deposits/studies/2027/2027v16`). It follows the read-only production sources and the version's `inputs/` files through the shared `deposit_utils` framework, notebooks 01 to 09, the sandbox `_2027v16` tables and the delivered workbooks, dashboards and promotion targets. Click a node to see its logic, parameters and lineage. The Run order tab walks the notebooks in sequence; the Issues tab lists findings from the review (2026-10-23), several confirmed with SQL against the sandbox tables.

This notebook only documents the study. It sits outside the version folder so it does not affect `lint.scan_version`.

In [0]:
# Deposit Study 2027v16 lineage graph definition.
# Built by reading the 2027v16 folder (docs, inputs, notebooks 01-09) and _shared/deposit_utils.
# Edit this cell to add/rename nodes; the render cell below picks up changes automatically.

LAYERS = [
    ("src",     "Production sources (read-only)"),
    ("inputs",  "Version inputs (inputs/, docs/)"),
    ("fw",      "Shared framework (deposit_utils)"),
    ("build",   "01 Build sandbox tables"),
    ("map",     "02 Product mapping"),
    ("beta",    "03 Beta engine"),
    ("legacy",  "04 Legacy / Empyrean betas"),
    ("report",  "05 Beta report"),
    ("deliv",   "Deliverables (outputs/)"),
    ("present", "06-08 Dashboards & compare"),
    ("final",   "Presentation, compare & promotion"),
]
L = {k: i for i, (k, _) in enumerate(LAYERS)}

NODES = []
def node(id, label, layer, kind, entity, desc, logic=None, params=None, warn=None, src=None, fq=None):
    NODES.append(dict(id=id, label=label, layer=L[layer], kind=kind, entity=entity, desc=desc,
                      logic=logic or [], params=params or {}, warn=warn, src=src, fq=fq or label))

NB = "studies/2027/2027v16/notebooks/"
SB = "finance.sandbox."

# ---------------------------------------------------------------- L0 production sources
node("p_pnfp_dep", "pnfp_deposits", "src", "table", "PNFP",
     "PNFP account-level deposit history (load_date, ledger, avgbal, rate, service charge code, override / prime rate numbers, flags). Token {{src_pnfp_deposits}}.",
     fq="finance.ftp_validation.pnfp_deposits")
node("p_lsnv", "lsnv_nmds", "src", "table", "SNV",
     "SNV account-level non-maturity deposits (subproductid, currcoupon, eopbook, avgbook, almAccountNum). Read straight from production by 01, 02, 04 and 07. Token {{src_lsnv_nmds}}.",
     fq="finance.pricing_analytics.lsnv_nmds")
node("p_ftp", "emp_ftp_results", "src", "table", "PNFP",
     "Empyrean FTP results. Links PNFP accounts to Empyrean AccountNum (acct_grp_nbr + first char of SecID) at ftp_results_date. Token {{src_emp_ftp_results}}.",
     fq="finance.ftp_validation.emp_ftp_results")
node("p_ffr", "fed_funds_matrix", "src", "table", "Both",
     "Daily fed funds: UP_FFR, LB_FFR, EFFR by effective_date. Token {{src_fed_funds}}.",
     fq="finance.operational_deposits.fed_funds_matrix")
node("p_dim", "reference.dim_nmd", "src", "table", "Both",
     "Production product dimension (subproductid, legacy_bank, pf_flag, bdsp_flag -> product_group, product, product_num). Exported once into inputs/product_maps/dim_nmd.csv. Token {{src_dim_nmd}}.",
     fq="finance.reference.dim_nmd")
node("p_nmds_agg", "pricing_analytics.nmds_agg", "src", "table", "Both",
     "Production product aggregate. Only used as the reconciliation baseline in 01 Step 6 ({{production_schema}}.nmds_agg).",
     fq="finance.pricing_analytics.nmds_agg")
node("p_lpnfp_prod", "pricing_analytics.lpnfp_nmds", "src", "table", "PNFP",
     "Production PNFP account table. Not part of the study flow; read only by a scratch %sql cell at the end of 05.",
     fq="finance.pricing_analytics.lpnfp_nmds")

# ---------------------------------------------------------------- L1 version inputs
node("i_study", "study.json", "inputs", "config", "Both",
     "Version identity. study_name 2027v16, parent none (created from templates 2026-10-16), status draft.",
     params={"study_name": "2027v16", "parent": "null", "status": "draft"}, src="2027v16/study.json")
node("i_params", "study_params.json", "inputs", "config", "Both",
     "Dates, schemas and source table names that every token resolves from.",
     params={"snapshot_date": "2026-06-30", "ftp_results_date": "2026-06-30", "chart_start_date": "2015-01-01",
             "sandbox_schema": "finance.sandbox", "production_schema": "finance.pricing_analytics"},
     src="inputs/config/study_params.json")
node("i_cycles", "rate_cycles.csv", "inputs", "config", "Both",
     "Rate-cycle windows. short_code drives the short column names (e.g. lm_0_TTC_PD1).",
     params={"TTC PD": "2019-01-31 to 2026-07-31", "TTC PD-1": "2019-01-31 to 2026-06-30", "125bpsdown (UP1)": "2019-07-31 to 2022-02-28",
             "500bpsUP (UP2)": "2022-03-31 to 2023-08-31", "MRT_DOWN (DN)": "2024-09-30 to 2026-06-30"},
     src="inputs/config/rate_cycles.csv")
node("i_pcfg", "product_config.csv", "inputs", "config", "Both",
     "Per-product beta method: typ AUTO / PURE / AVG_M / MAN / EMP, methods m1/m2 (LR, POLY), lag, cycle, manual beta. 32 product rows.",
     params={"AUTO": "best R2 across all methods (4 NIB DDA rows)", "MAN": "Indexed = 1.0, Savings Commercial 0.2, Savings Consumer / PF 0.05",
             "EMP": "IB DDA Hybrid Public Funds", "AVG_M": "average of POLY and LR", "Standard Plus row": "PURE LR lag 1 TTC"},
     src="inputs/config/product_config.csv")
node("i_promo", "promotion.csv", "inputs", "config", "Both",
     "Promotion plan: sandbox table -> target, include Y/N, mode. All five rows are include = N.",
     params={"nmds_agg / lpnfp_nmds / lpnfp_nmds_agg / lsnv_nmds_agg": "finance.pricing_analytics.*", "dim_nmd": "finance.reference.dim_nmd (owned outside the team)"},
     src="inputs/config/promotion.csv")
node("i_dimcsv", "dim_nmd.csv", "inputs", "file", "Both",
     "This version's copy of the product dimension. Editing it reclassifies products; re-run from 01.", src="inputs/product_maps/dim_nmd.csv")
node("i_lb_snv", "legacy_betas_snv.csv", "inputs", "file", "SNV",
     "SNV legacy betas per Empyrean account: BetaCalcType, CalculatedBeta, RSquare, EmpyreanModelBeta, SuggestedBeta (35 rows).", src="inputs/betas/legacy_betas_snv.csv")
node("i_lb_pnfp", "legacy_betas_pnfp.csv", "inputs", "file", "PNFP",
     "PNFP legacy betas, same layout (22 rows).", src="inputs/betas/legacy_betas_pnfp.csv")
node("i_trunc", "truncations.csv", "inputs", "file", "Both",
     "Truncation point and predicted 30 / 25 / 20 percent truncations by product group (48 rows).", src="inputs/betas/truncations.csv")
node("i_pnfp_map", "pnfp_acct_map.csv", "inputs", "file", "PNFP",
     "PNFP account type -> deposit group and segment (229 rows).", src="inputs/product_maps/pnfp_acct_map.csv")
node("i_pnfp_prod", "pnfp_products.csv", "inputs", "file", "PNFP",
     "PNFP subproduct attributes: deposit group, segment, rate class, pf_flag, ALM account name, GL account, level 6 hierarchy.", src="inputs/product_maps/pnfp_products.csv")
node("i_snv_map", "snv_acct_map.csv", "inputs", "file", "SNV",
     "SNV account type -> deposit group and segment (124 rows).", src="inputs/product_maps/snv_acct_map.csv")
node("i_snv_prod", "snv_product_map.csv", "inputs", "file", "SNV",
     "SNV account type code -> product number (127 rows).", src="inputs/product_maps/snv_product_map.csv")
node("i_future", "future_products_map.csv", "inputs", "file", "Both",
     "Legacy account type -> recommended future-state account type, product type and product group (185 rows).", src="inputs/product_maps/future_products_map.csv")
node("i_changelog", "change_log.md", "inputs", "file", "Both",
     "Version change log. Only entry: 2026-10-16 created 2027v16 from templates.", src="docs/change_log.md")

# ---------------------------------------------------------------- L2 shared framework
node("f_study", "Study (S)", "fw", "function", "Both",
     "Study.load() finds the version folder from the notebook path, reads study.json and study_params.json and exposes the tag, dates, S.table(), S.input(), S.output(), S.cycles and the token map.",
     ["Bootstrap cell walks up to _shared/deposit_utils and inserts it on sys.path", "S.table('x') -> finance.sandbox.x_2027v16", "S.output(kind, name, ext) appends _2027v16 to the file name"],
     src="_shared/deposit_utils/study.py")
node("f_sql", "sql() / sqlq()", "fw", "function", "Both",
     "Runs multi-statement SQL with {{token}} substitution (sql) or returns a DataFrame (sqlq).", src="_shared/deposit_utils/sqlrun.py")
node("f_spark_io", "spark_io", "fw", "function", "Both",
     "I/O helpers: seed_dim_nmd_csv, load_dim_nmd_table, csv_to_view, alias_view, persist_view.", src="_shared/deposit_utils/spark_io.py")
node("f_betas", "betas.py", "fw", "function", "Both",
     "Beta engine: create_pg_products_view, load_war_data, load_effr_monthly, build_named_cycles, endpoint / lm / poly beta functions, run_all_betas, save / load_final_results.",
     ["Linear: OLS WAR ~ UP_FFR, beta = coef x 100, R2 x 100", "Endpoint: (WAR_end - WAR_start) / (FFR_end - FFR_start) x 100", "Poly: cubic fit, beta = derivative at end-of-window FFR (needs >= 4 points)", "Lag n: WAR window starts n months later, FFR window ends n months earlier"],
     src="_shared/deposit_utils/betas.py")
node("f_config", "config.py", "fw", "function", "Both",
     "ProductConfig.from_study reads product_config.csv; validate() flags unknown methods, cycles or lags; METHOD_LOOKUP / CALC_TYPE_LOOKUP.", src="_shared/deposit_utils/config.py")
node("f_compare", "compare.py", "fw", "function", "Both",
     "compare_versions(root_a, root_b): input diffs, mapping and method changes, beta / WAR / balance deltas from the two beta_summary files.", src="_shared/deposit_utils/compare.py")
node("f_promote", "promote.py", "fw", "function", "Both",
     "load_plan, dry_run, execute (backup first, typed confirmation), restore_command.", src="_shared/deposit_utils/promote.py")
node("f_folders", "folders.py", "fw", "function", "Both",
     "ensure_schema, list_versions, tag_path.", src="_shared/deposit_utils/folders.py")
node("f_lint", "lint.scan_version", "fw", "function", "Both",
     "Scans a version's notebooks for typed table names, dates, paths and version tags. On 2026-10-23 it reports 2 findings in 2027v16, both in the 05 scratch cell.", src="_shared/deposit_utils/lint.py")

# ---------------------------------------------------------------- L3 01 build tables
node("nb01", "01_build_tables", "build", "notebook", "Both",
     "Builds this version's sandbox dim_nmd, PNFP and SNV aggregates and nmds_agg, then reconciles to production. Last run 2026-10-19 11:52-11:53.", src=NB + "01_build_tables")
node("b_dim", "dim_nmd_2027v16", "build", "table", "Both",
     "Sandbox product dimension loaded from dim_nmd.csv with production column types (Step 2).", fq=SB + "dim_nmd_2027v16")
node("b_pnfp_s", "lpnfp_nmds_s", "build", "view", "PNFP",
     "Temp view over pnfp_deposits that derives the PNFP subproductid (Step 3).",
     ["accountid = account_type || account_number; eopbook = ledger; avgbook = avgbal; currcoupon = rate",
      "dnc_flag = 'Y' when prime or override rate number is 787 (OPTION 1, on)",
      "suffix 000_01 when override in (590, 786, 2, 791, 789)",
      "suffix 000_00 when override in ('0', '', '100') or prime = '999', and override <> '787' (OPTION 2a on)",
      "explicit 787 -> 082_00 branch (OPTION 2c, on) then ELSE 082_00",
      "subproductid = service_charge_code || '_' || suffix"],
     warn="Standard Plus OPTION 1, 2a and 2c are switched on here and OPTION 3 in lpnfp_nmds, but docs/change_log.md only records the version's creation. OPTION 2c sits below the 000_00 branch, so it changes nothing; prime-787 accounts with override 0 still fall into 000_00 (see nmds_agg_2027v16).",
     src=NB + "01_build_tables (Step 3)")
node("b_lpnfp", "lpnfp_nmds_2027v16", "build", "table", "PNFP",
     "PNFP account-level table: lpnfp_nmds_s joined to dim_nmd on subproductid + pf_flag, legacy_bank PNFP, bdsp_flag N.",
     ["product = 'IB DDA Commercial Standard Plus' when subproductid ends 082_00, dnc_flag = 'N' and dim product is 'IB DDA Commercial Exception' (OPTION 3)", "else dim_nmd product"],
     warn="OPTION 3 is narrower than docs/optional_logic.md describes: the rename is limited by a typed product name (IB DDA Commercial Exception), so Consumer and MMA Exception accounts are never renamed. Confirm this is the intended rate-class rule.",
     fq=SB + "lpnfp_nmds_2027v16")
node("b_lpnfp_agg", "lpnfp_nmds_agg_2027v16", "build", "table", "PNFP",
     "PNFP aggregate by eopdate, acct_typ_desc, subproductid, product_group, product, product_num. Excludes internal_flag = '1' and closed_flag = '1'.", fq=SB + "lpnfp_nmds_agg_2027v16")
node("b_snv_v", "snv_nmds", "build", "view", "SNV",
     "Temp view over lsnv_nmds adding bdsp_flag and sumprod columns (Step 4).",
     ["bdsp_flag = 'Y' for accounts 16501007815721 and 16501007815713", "sumprod_eoprate = currcoupon x eopbook"],
     warn="The two BDSP account ids and the excluded subproducts (050_082_00, 050_000_00) are typed into notebooks 01, 02 and 07 instead of an inputs/ file, so three copies have to be kept in step.",
     src=NB + "01_build_tables (Step 4)")
node("b_lsnv_agg", "lsnv_nmds_agg_2027v16", "build", "table", "SNV",
     "SNV aggregate: snv_nmds joined to dim_nmd on subproductid + bdsp_flag (SNV), excluding 050_082_00 and 050_000_00.", fq=SB + "lsnv_nmds_agg_2027v16")
node("b_nmds_agg", "nmds_agg_2027v16", "build", "table", "Both",
     "UNION ALL of the SNV and PNFP aggregates. The single source for WAR history, betas and dashboards.",
     warn="At 2026-06-30, $15,163,601.84 has a null product_group: 31 PNFP accounts in 3D_000_00 ($15,163,381) and 1 in 4F_000_00 ($220), all prime rate 787 with override 0. dim_nmd has no 000_00 row for 3D or 4F, and pg_products drops product_group IS NULL, so these balances are left out of every beta. The 01 reconciliation shows the same gap (IB DDA -$15.16M, MMA -$220 vs production). Verified with SQL on 2026-10-23.",
     fq=SB + "nmds_agg_2027v16")
node("b_recon", "Step 6 reconciliation", "build", "query", "Both",
     "Row count, balance and WAR vs production nmds_agg at snapshot_date, then balance difference by product group (FULL OUTER JOIN).", src=NB + "01_build_tables (Step 6)")

# ---------------------------------------------------------------- L4 02 product mapping
node("nb02", "02_product_mapping", "map", "notebook", "Both",
     "Legacy account types -> future-state products with rate statistics, an HTML dashboard and an Excel workbook. Last run 2026-10-19.", src=NB + "02_product_mapping")
node("m_pnfp_map", "pnfp_acct_map", "map", "view", "PNFP",
     "PNFP account map from CSV with deposit_type and product_type derived from deposit_group.", src=NB + "02_product_mapping (Step 2)")
node("m_pnfp_s", "PNFP_S", "map", "view", "PNFP",
     "PNFP subproduct attribute view from pnfp_products.csv. Its downstream use was not traced in this review.", src=NB + "02_product_mapping (Step 2)")
node("m_snv_maps", "snv_acct_map / snv_product_map", "map", "view", "SNV",
     "SNV account and product maps exposed as views (Step 3).")
node("m_pnfp_base", "pnfp_deposits_base", "map", "view", "PNFP",
     "PNFP account-level base from lpnfp_nmds_2027v16 with wa_rate = currcoupon x eopbook, plus metrics by account type.")
node("m_snv_base", "snv_deposits_base", "map", "view", "SNV",
     "SNV account-level base. OPTION S is on: s1 adds bdsp_flag and the base joins dim_nmd to carry product.",
     ["s1 = lsnv_nmds + bdsp_flag", "LEFT JOIN dim_nmd on subproductid, legacy_bank SNV, bdsp_flag"])
node("m_combined", "combined_deposits_base", "map", "view", "Both",
     "UNION of both banks joined to their account maps (deposit_group, segment, product_type, deposit_type, product).",
     warn="OPTION P is only partly on: product is selected here and rate_analysis_by_product groups by product, but the filter still uses deposit_group IS NOT NULL (not the documented Brokered filter) and the Step 7 dashboard still groups by recommended_product_group. optional_logic.md says to switch these on together. None of it is in the change log.")
node("m_rate", "rate_analysis_by_product", "map", "view", "Both",
     "Account count, balances, WAR, P25 / P50 / P75 / max rate and balance share by bank, eopdate and product. Excludes Internal segment.")
node("m_future", "future_productsmap", "map", "view", "Both",
     "Future-state product map view from future_products_map.csv.")
node("m_dash", "mapping dashboard builder", "map", "function", "Both",
     "At snapshot_date: combined base LEFT JOIN future map on bank + account type; unmapped rows show 'Unmapped'. Groups by new code and writes the HTML dashboard and the mapping workbook.", src=NB + "02_product_mapping (Step 7)")

# ---------------------------------------------------------------- L5 03 beta engine
node("nb03", "03_beta_models", "beta", "notebook", "Both",
     "Endpoint, linear and cubic betas for every product, cycle and lag 0-3. Last run 2026-10-19 11:57.", src=NB + "03_beta_models")
node("e_pg", "pg_products", "beta", "view", "Both",
     "Temp view over nmds_agg grouped by EOPDATE, PRODUCT_GROUP, PRODUCT: balances and WAR = sum(sumprod_eoprate) / sum(eopbook). Filters product_group IS NOT NULL.", src="_shared/deposit_utils/betas.py")
node("e_war", "war_pivot / acct_to_pg", "beta", "view", "Both",
     "Collapses to unique (product, month) with balance-weighted WAR, pivots to one row per product. 37 products across 12 product groups, 2017-12-31 to 2026-07-31.")
node("e_effr", "effr_monthly", "beta", "view", "Both",
     "Fed funds resampled to month-end (last observation per month).")
node("e_cycles", "NAMED_CYCLES", "beta", "view", "Both",
     "(cycle name, month-end dates) per rate_cycles.csv row: TTC PD 91 months, TTC PD-1 90, 125bpsdown 32, 500bpsUP 18, MRT_DOWN 22.")
node("e_methods", "run_all_betas", "beta", "function", "Both",
     "5 cycles x 4 lags x (endpoint, linear on UP_FFR, cubic on UP_FFR, cubic on EFFR).",
     ["Output columns: Endpoint_Beta / Lm_Coef / R_Square / Poly_Beta / Poly_R_Square / Poly_Beta_FFE / Poly_R_Square_FFE Lag n <cycle>"])
node("e_final", "final_results", "beta", "view", "Both", "Wide DataFrame with long column names (spaces), one row per product.")
node("e_short", "short_results", "beta", "view", "Both", "Same results with _shorten() names, e.g. lm_0_TTC_PD, pr2e_3_TTC_PD.")
node("e_chk", "Step 5 sanity check", "beta", "query", "Both", "Products missing lm_0_TTC_PD: 0 of 37 on the 2026-10-19 run.")
node("o_final_csv", "final_results_2027v16.csv", "beta", "file", "Both",
     "Long-name betas (Delta cannot store names with spaces). Read by 05 and 06.", fq="outputs/intermediate/final_results_2027v16.csv")
node("o_ccalc", "ccalc_beta_results_2027v16", "beta", "table", "Both",
     "Short-name betas. Read by the 04 SQL and 05.", fq=SB + "ccalc_beta_results_2027v16")

# ---------------------------------------------------------------- L6 04 legacy betas
node("nb04", "04_legacy_betas", "legacy", "notebook", "Both",
     "Joins legacy Empyrean / suggested betas, truncations and this version's calculated betas to legacy accounts, then rolls up by product. Last run 2026-10-19 12:00.", src=NB + "04_legacy_betas")
node("g_trunc", "truncation_values", "legacy", "view", "Both", "truncations.csv as a view (48 rows).")
node("g_snv_bv", "snv_beta_values", "legacy", "view", "SNV", "legacy_betas_snv.csv as a view; blanks -> null (35 rows).")
node("g_pnfp_bv", "pnfp_beta_values", "legacy", "view", "PNFP", "legacy_betas_pnfp.csv as a view (22 rows).")
node("g_lb", "legacy_betas", "legacy", "view", "Both",
     "Both banks' legacy betas UNION ALL, LEFT JOIN truncations on UPPER(TRIM(product)) = UPPER(TRIM(product_group)).")
node("g_pnfp_beta", "pnfp_beta", "legacy", "view", "PNFP",
     "lpnfp_nmds INNER JOIN emp_ftp_results on acct_grp_nbr = SUBSTRING(accountid, 2), first char of SecID, EOPDATE = ftp_results_date.",
     warn="The INNER JOIN drops PNFP accounts with no FTP match: $1.148B of the $47.25B in-scope balance at 2026-06-30, so legacy_consolid_betas carries $46.10B for PNFP. It also does not filter internal or closed accounts (none matched in this run). Verified with SQL on 2026-10-23.")
node("g_pnfp_det", "pnfp_detail", "legacy", "view", "PNFP",
     "PNFP accounts LEFT JOIN legacy_betas on AccountNum at snapshot_date: balances, WAR, Empyrean / suggested beta, BetaCalcType. (pnfp_betalow is a similar inner-join view that is not used downstream.)")
node("g_lpnfp_betas", "lpnfp_betas", "legacy", "view", "PNFP",
     "Balance-weighted Empyrean and suggested betas by product and legacy account, plus the matching calculated betas.",
     ["preferred_ccalc_beta = ccalc column picked by BetaCalcType (ed, lm_0..lm_3) on TTC_PD1", "best / second R2 method ranked over lm_0..lm_3 on TTC_PD1", "preferred poly beta and R2 (UP_FFR and EFFR) at the BetaCalcType lag"],
     warn="best_r2 is joined with AND best_r2_method = BetaCalcType, so best and second R2 columns are null unless the legacy method already is the best one: 85 of 113 legacy_consolid_betas rows (PNFP 50/66, SNV 35/47). The join probably should be on EMP_NAME only. Same SQL in lsnv_betas. Verified with SQL on 2026-10-23.")
node("g_snv_beta", "snv_beta", "legacy", "view", "SNV", "Production lsnv_nmds at snapshot_date.")
node("g_snv_det", "snv_detail", "legacy", "view", "SNV",
     "snv_beta JOIN legacy_betas on almAccountNum, JOIN lsnv_nmds_agg_2027v16 on subproductid + eopdate for product and WAR. No duplicate subproduct rows existed in the agg at 2026-06-30.")
node("g_lsnv_betas", "lsnv_betas", "legacy", "view", "SNV",
     "SNV roll-up, same columns as lpnfp_betas.",
     warn="The preferred, best-R2 and poly columns in lsnv_betas and lpnfp_betas hardcode the TTC_PD1 cycle code. They ignore the product_config.csv cycle and break if the short_code in rate_cycles.csv changes.")
node("g_consol", "legacy_consolid_betas", "legacy", "view", "Both", "lpnfp_betas UNION ALL lsnv_betas with legacy_bank.")
node("o_lb_tbl", "legacy_betas_2027v16", "legacy", "table", "Both", "Persisted legacy_betas (57 rows).", fq=SB + "legacy_betas_2027v16")
node("o_consol_tbl", "legacy_consolid_betas_2027v16", "legacy", "table", "Both",
     "Persisted roll-up (113 rows): PNFP 66 rows / 32 products / $46.10B, SNV 47 rows / 25 products / $38.22B.", fq=SB + "legacy_consolid_betas_2027v16")
node("g_chk", "Step 8 checks", "legacy", "query", "Both", "Rows, products, balance and rows without an Empyrean beta per bank (10 PNFP, 10 SNV).")

# ---------------------------------------------------------------- L7 05 beta report
node("nb05", "05_beta_report", "report", "notebook", "Both",
     "Applies product_config and writes the beta workbooks and beta_summary.",
     warn="Last run 2026-10-16 21:22, but 01 to 04 re-ran on 2026-10-19. The workbooks and beta_summary (CSV and table) come from older betas: 05 saw 36 products, 03 now has 37. Re-run 05, then 08 if comparing.",
     src=NB + "05_beta_report")
node("r_load", "Step 1 load", "report", "function", "Both",
     "alias_view for legacy_betas, legacy_consolid_betas, ccalc_beta_results; rebuilds war_pivot, effr_monthly, NAMED_CYCLES; loads final_results CSV.")
node("r_pc", "ProductConfig + validate", "report", "function", "Both",
     "Reads product_config.csv, validates methods / cycles / lags against acct_to_pg, lists configured vs auto-select products.",
     warn="On the 2026-10-16 run 'IB DDA Commercial Standard Plus' had no product_config row and defaulted to AUTO (36 products, 25 configured, 11 auto). The CSV now has PURE LR lag 1 TTC for it, which takes effect only after 05 re-runs.")
node("r_detail", "Beta Detail (tree)", "report", "function", "Both",
     "Top 3 regression methods and top polynomial per product, each with the average of beta and Empyrean; builds tree (Step 3).")
node("r_exec", "Executive summary workbook", "report", "function", "Both",
     "Step 4: Executive Summary, Internal & Other, BetaCalcType, Account Detail, Truncations, Beta Detail tabs; defines wc() and styles.",
     warn="The output name is typed as 'v16_Beta_Analysis_by_Product', which gives v16_Beta_Analysis_by_Product_2027v16.xlsx. A version cloned from 2027v16 keeps the 'v16_' prefix, and outputs_guide.md expects Beta_Analysis_by_Product_<tag>. lint does not catch it.")
node("r_pgwb", "PG workbook", "report", "function", "Both",
     "Step 5: one tab per product group, color scales on R2 and difference from Empyrean.")
node("r_cfgdata", "config_data / recommended beta", "report", "function", "Both",
     "Step 6: picks the beta per product and adds Configuration, Product, PG and Bank summary tabs.",
     ["EMP -> Empyrean beta; MAN -> manual value", "AVG_M (blend_poly_lm) -> mean of poly and LR betas and R2s", "PURE -> configured column", "AUTO -> best R2 over all cycles, methods and lags (endpoint only if nothing else)",
      "recommended = calc beta, except AVG_E = (calc + Empyrean) / 2 and EMP = Empyrean", "WAR and balances at snapshot_month; Brokered DDA / MMA excluded"],
     warn="CYCLE_DISPLAY keys ('Up Cycle 19-22', 'Up Cycle 22-23', 'Down Cycle 24-26') do not match the cycle names in rate_cycles.csv (125bpsdown, 500bpsUP, MRT_DOWN), so AUTO rows show raw names. The AVG_E branch uses 'cb or emp_f', which treats a 0 beta as missing.")
node("r_save", "Step 7 save beta_summary", "report", "function", "Both",
     "One row per product: study, method, cycle, R2, Empyrean / calc / recommended beta, WAR, balances, calc_type, snapshot_date.")
node("r_trace", "Trace a product", "report", "query", "Both",
     "Counts rows matching a product name ('Standard Plus') at every stage from lpnfp_nmds to config_data; flags rows with no calculated beta.")
node("r_scratch", "scratch %sql cell", "report", "query", "PNFP",
     "Ad-hoc look at IB DDA Commercial Exception accounts in production lpnfp_nmds.",
     warn="Typed production table and date (finance.pricing_analytics.lpnfp_nmds, '2026-06-30'): it reads production, not this version, and these are the 2 lint findings. The filter 'prime_rate <> 787 OR override_rate <> 787' is true unless both are 787; AND was probably meant.")

# ---------------------------------------------------------------- L8 deliverables
node("o_bs_tbl", "beta_summary_2027v16", "deliv", "table", "Both", "Recommended betas as a sandbox table.", fq=SB + "beta_summary_2027v16")
node("o_bs_csv", "beta_summary_2027v16.csv", "deliv", "file", "Both", "Recommended betas flat file; what 08_version_compare reads.", fq="outputs/flatfiles/beta_summary_2027v16.csv")
node("o_xl_beta", "v16_Beta_Analysis_by_Product_2027v16.xlsx", "deliv", "file", "Both", "Executive summary and detail workbook.", fq="outputs/excel/v16_Beta_Analysis_by_Product_2027v16.xlsx")
node("o_xl_consol", "Consolidated_Beta_Report_2027v16.xlsx", "deliv", "file", "Both",
     "The deliverable: Configuration, Product / PG / Bank summaries and one tab per product group.", fq="outputs/excel/Consolidated_Beta_Report_2027v16.xlsx")
node("o_xl_map", "Deposit_Product_Mapping_2027v16.xlsx", "deliv", "file", "Both", "Legacy to future-state mapping with rate statistics.", fq="outputs/excel/Deposit_Product_Mapping_2027v16.xlsx")
node("o_html_map", "Product_Mapping_Dashboard_2027v16.html", "deliv", "file", "Both", "Standalone mapping dashboard.", fq="outputs/html/Product_Mapping_Dashboard_2027v16.html")

# ---------------------------------------------------------------- L9 06-08
node("nb06", "06_dashboards", "present", "notebook", "Both", "Fed funds chart and interactive product-group dashboard.", src=NB + "06_dashboards")
node("nb07", "07_account_to_product", "present", "notebook", "Both", "Account-level WAR percentiles by product and subproduct at snapshot_date.", src=NB + "07_account_to_product")
node("nb08", "08_version_compare", "present", "notebook", "Both", "Compares two versions picked in widgets (A defaults to the parent or latest other version).", src=NB + "08_version_compare")
node("x_ffr", "fed funds cycle chart", "present", "function", "Both", "Rate-change dates from UP_FFR, contiguous hike / cut cycles from chart_start_date, slope and total bps per cycle.")
node("x_pgdash", "PG dashboard builder", "present", "function", "Both", "Long-format betas from final_results, WAR series from war_pivot, latest stats per product.")
node("x_a2p", "account WAR percentiles", "present", "query", "Both",
     "SNV (lsnv_nmds + dim_nmd, BDSP rule, excluded subproducts) UNION PNFP (lpnfp_nmds, no internal / closed) at snapshot_date; percentiles, std dev, WAR by group / product / subproduct.")
node("x_compare", "compare_versions", "present", "function", "Both", "Input, mapping, method and beta differences between version A and B.")

# ---------------------------------------------------------------- L10 final
node("o_html_pg", "Interactive_PG_Dashboard_<tag>.html", "final", "file", "Both",
     "Product-group beta dashboard; outputs_guide.md says to present this first.",
     warn="06 and 07 have not been run in 2027v16: outputs/ has no Interactive_PG_Dashboard, Fed_Funds_Cycles or Account_to_Product file. 08 has not been run either.",
     fq="outputs/html/Interactive_PG_Dashboard_2027v16.html (not produced)")
node("o_html_ffr", "Fed_Funds_Cycles_<tag>.html", "final", "file", "Both", "Fed funds history with cycles (not produced yet).", fq="outputs/html/Fed_Funds_Cycles_2027v16.html")
node("o_xl_a2p", "Account_to_Product_Analysis_<tag>.xlsx", "final", "file", "Both", "Account-level WAR percentiles (not produced yet).", fq="outputs/excel/Account_to_Product_Analysis_2027v16.xlsx")
node("o_cmp", "<A>_vs_<B>.xlsx", "final", "file", "Both", "Comparison workbook in <year>/comparisons.", fq="studies/2027/comparisons/<A>_vs_<B>.xlsx")
node("o_allv", "beta_summary_all_versions", "final", "view", "Both", "Cross-version view of every version's beta_summary.", fq=SB + "beta_summary_all_versions")
node("nb09", "09_promote_to_pricing_analytics", "final", "notebook", "Both",
     "Copies final sandbox tables to the targets in promotion.csv. Never run in 2027v16.", src=NB + "09_promote_to_pricing_analytics")
node("z_dry", "dry_run", "final", "function", "Both", "Row counts vs current target, schema match, blocking issues. Writes nothing.")
node("z_exec", "execute", "final", "function", "Both",
     "Runs only with Mode = execute and the typed text 'PROMOTE 2027v16'; backs up each target into the sandbox first.",
     params={"replace": "CREATE OR REPLACE TABLE ... AS SELECT", "overwrite": "INSERT OVERWRITE (fails on schema change)"})
node("t_prod", "pricing_analytics targets", "final", "external", "Both",
     "finance.pricing_analytics.nmds_agg, lpnfp_nmds, lpnfp_nmds_agg, lsnv_nmds_agg. Shared with other teams' reports; all include = N.", fq="finance.pricing_analytics.{nmds_agg, lpnfp_nmds, lpnfp_nmds_agg, lsnv_nmds_agg}")
node("t_ref", "reference.dim_nmd target", "final", "external", "Both", "Owned outside the team; confirm with the owner before promoting.", fq="finance.reference.dim_nmd")
node("o_promo_log", "promotion_log_2027v16.csv", "final", "file", "Both", "Every promotion attempt and backup name.", fq="outputs/logs/promotion_log_2027v16.csv")

# ---------------------------------------------------------------- edges: d data, o run/call, p parameter/config, r recon/check
EDGES = []
def e(s, t, type="d", label=None):
    EDGES.append(dict(s=s, t=t, type=type, label=label))

# framework and run order
e("i_study", "f_study", "p", "tag"); e("i_params", "f_study", "p", "dates, sources")
for nb in ["nb01", "nb02", "nb03", "nb04", "nb05", "nb06", "nb07", "nb08", "nb09"]:
    e("f_study", nb, "o", "Study.load")
e("f_sql", "nb01", "o", "token SQL"); e("f_folders", "nb01", "o", "ensure_schema"); e("f_folders", "nb08", "o", "list_versions")
e("f_lint", "f_study", "r", "scan_version(S.root)")
for a, b in [("nb01", "nb02"), ("nb01", "nb03"), ("nb03", "nb04"), ("nb04", "nb05"), ("nb03", "nb06"), ("nb01", "nb07"), ("nb05", "nb08"), ("nb05", "nb09")]:
    e(a, b, "o", "run after")

# 01
e("nb01", "b_dim", "o"); e("nb01", "b_pnfp_s", "o")
e("p_dim", "i_dimcsv", "d", "seed on first run"); e("i_dimcsv", "b_dim", "d", "load_dim_nmd_table"); e("f_spark_io", "b_dim", "o", "seed / load")
e("p_pnfp_dep", "b_pnfp_s"); e("b_pnfp_s", "b_lpnfp"); e("b_dim", "b_lpnfp", "d", "subproductid + pf_flag")
e("b_pnfp_s", "i_changelog", "r", "options not logged")
e("b_lpnfp", "b_lpnfp_agg", "d", "excl internal / closed")
e("p_lsnv", "b_snv_v"); e("b_snv_v", "b_lsnv_agg"); e("b_dim", "b_lsnv_agg", "d", "subproductid + bdsp_flag")
e("b_lpnfp_agg", "b_nmds_agg"); e("b_lsnv_agg", "b_nmds_agg")
e("b_nmds_agg", "b_recon", "r"); e("p_nmds_agg", "b_recon", "r", "production baseline")

# 02
e("nb02", "m_pnfp_map", "o"); e("f_spark_io", "m_pnfp_map", "o", "csv_to_view")
e("i_pnfp_map", "m_pnfp_map"); e("i_pnfp_prod", "m_pnfp_s"); e("i_snv_map", "m_snv_maps"); e("i_snv_prod", "m_snv_maps"); e("i_future", "m_future")
e("b_lpnfp", "m_pnfp_base"); e("p_lsnv", "m_snv_base"); e("b_dim", "m_snv_base", "d", "OPTION S join")
e("m_pnfp_base", "m_combined"); e("m_snv_base", "m_combined"); e("m_pnfp_map", "m_combined", "d", "acct type"); e("m_snv_maps", "m_combined", "d", "acct type")
e("m_combined", "m_rate"); e("m_combined", "m_dash"); e("m_future", "m_dash", "d", "bank + acct type")
e("i_params", "m_dash", "p", "snapshot_date")
e("m_dash", "o_html_map"); e("m_dash", "o_xl_map"); e("m_rate", "o_xl_map", "d", "rate stats"); e("m_pnfp_s", "o_xl_map", "d", "inferred")

# 03
e("nb03", "e_war", "o"); e("f_betas", "e_pg", "o"); e("f_betas", "e_methods", "o")
e("b_nmds_agg", "e_pg", "d", "product_group IS NOT NULL"); e("e_pg", "e_war")
e("p_ffr", "e_effr", "d", "month-end"); e("e_effr", "e_cycles"); e("i_cycles", "e_cycles", "p", "windows")
e("e_war", "e_methods"); e("e_cycles", "e_methods"); e("e_effr", "e_methods", "d", "UP_FFR / EFFR")
e("e_methods", "e_final"); e("e_methods", "e_short", "d", "_shorten")
e("e_final", "o_final_csv", "d", "save_final_results"); e("e_short", "o_ccalc", "d", "overwrite"); e("e_short", "e_chk", "r")

# 04
e("nb04", "g_lb", "o"); e("f_spark_io", "g_trunc", "o", "csv_to_view"); e("f_spark_io", "o_consol_tbl", "o", "persist_view")
e("i_trunc", "g_trunc"); e("i_lb_snv", "g_snv_bv"); e("i_lb_pnfp", "g_pnfp_bv")
e("g_snv_bv", "g_lb"); e("g_pnfp_bv", "g_lb"); e("g_trunc", "g_lb", "d", "product = product_group")
e("b_lpnfp", "g_pnfp_beta"); e("p_ftp", "g_pnfp_beta", "d", "INNER JOIN"); e("i_params", "g_pnfp_beta", "p", "ftp_results_date")
e("g_pnfp_beta", "g_pnfp_det"); e("g_lb", "g_pnfp_det", "d", "AccountNum")
e("g_pnfp_det", "g_lpnfp_betas"); e("o_ccalc", "g_lpnfp_betas", "d", "EMP_NAME = ACCT_PRODUCT")
e("p_lsnv", "g_snv_beta", "d", "snapshot_date"); e("g_snv_beta", "g_snv_det"); e("g_lb", "g_snv_det", "d", "almAccountNum"); e("b_lsnv_agg", "g_snv_det", "d", "subproductid + eopdate")
e("g_snv_det", "g_lsnv_betas"); e("o_ccalc", "g_lsnv_betas", "d", "EMP_NAME = ACCT_PRODUCT")
e("g_lpnfp_betas", "g_consol"); e("g_lsnv_betas", "g_consol")
e("g_lb", "o_lb_tbl", "d", "persist"); e("g_consol", "o_consol_tbl", "d", "persist"); e("o_consol_tbl", "g_chk", "r")

# 05
e("nb05", "r_load", "o"); e("f_betas", "r_load", "o", "load_war_data"); e("f_config", "r_pc", "o")
e("o_lb_tbl", "r_load"); e("o_consol_tbl", "r_load"); e("o_ccalc", "r_load"); e("o_final_csv", "r_load"); e("b_nmds_agg", "r_load", "d", "pg_products")
e("i_pcfg", "r_pc", "p", "method / lag / cycle"); e("r_load", "r_pc", "d", "acct_to_pg")
e("r_load", "r_detail"); e("r_pc", "r_detail", "p")
e("r_detail", "r_exec"); e("r_exec", "o_xl_beta")
e("r_detail", "r_pgwb", "d", "tree"); e("r_exec", "r_cfgdata", "d", "summary_main, styles"); e("r_pgwb", "r_cfgdata", "d", "workbook"); e("r_pc", "r_cfgdata", "p", "resolve_config")
e("r_pgwb", "o_xl_consol"); e("r_cfgdata", "o_xl_consol", "d", "summary tabs")
e("r_cfgdata", "r_save"); e("r_save", "o_bs_csv"); e("r_save", "o_bs_tbl")
for s in ["b_lpnfp", "b_nmds_agg", "o_ccalc", "o_consol_tbl"]:
    e(s, "r_trace", "r")
e("p_lpnfp_prod", "r_scratch", "d", "typed table + date")

# 06-08
e("nb06", "x_ffr", "o"); e("nb06", "x_pgdash", "o"); e("nb07", "x_a2p", "o"); e("nb08", "x_compare", "o"); e("f_compare", "x_compare", "o")
e("p_ffr", "x_ffr"); e("i_params", "x_ffr", "p", "chart_start_date")
e("o_final_csv", "x_pgdash"); e("b_nmds_agg", "x_pgdash", "d", "war_pivot")
e("p_lsnv", "x_a2p"); e("b_lpnfp", "x_a2p"); e("b_dim", "x_a2p", "d", "SNV classification")
e("o_bs_csv", "x_compare", "d", "A and B"); e("i_pcfg", "x_compare", "p", "method diff"); e("i_dimcsv", "x_compare", "p", "mapping diff"); e("i_cycles", "x_compare", "p", "cycle diff")

# final
e("x_ffr", "o_html_ffr"); e("x_pgdash", "o_html_pg"); e("x_a2p", "o_xl_a2p"); e("x_compare", "o_cmp"); e("x_compare", "o_allv")
e("nb09", "z_dry", "o"); e("f_promote", "z_dry", "o"); e("f_promote", "z_exec", "o"); e("i_promo", "z_dry", "p", "plan")
e("z_dry", "z_exec", "o", "execute + PROMOTE <tag>")
for s in ["b_nmds_agg", "b_lpnfp", "b_lpnfp_agg", "b_lsnv_agg", "b_dim"]:
    e(s, "z_dry", "d", "sandbox table")
e("z_exec", "t_prod", "d", "include = Y only"); e("z_exec", "t_ref", "d", "include = Y only"); e("z_exec", "o_promo_log")

# ---------------------------------------------------------------- run-order stages
STAGES = [
    ("Bootstrap", "Every notebook finds _shared, loads Study from study.json and study_params.json; tokens resolve tables and dates.",
     ["i_study", "i_params", "f_study", "f_sql", "f_lint"]),
    ("01 Product dimension", "dim_nmd.csv (seeded from production once) loaded into the sandbox.", ["p_dim", "i_dimcsv", "b_dim", "f_spark_io"]),
    ("01 PNFP classification", "Subproduct from service charge + override / prime rate, Standard Plus options, dim join, aggregate.",
     ["p_pnfp_dep", "b_pnfp_s", "b_lpnfp", "b_lpnfp_agg", "i_changelog"]),
    ("01 SNV aggregate", "BDSP flag, dim join, excluded subproducts.", ["p_lsnv", "b_snv_v", "b_lsnv_agg"]),
    ("01 Combine and reconcile", "nmds_agg = SNV + PNFP; compare to production at snapshot_date.", ["b_nmds_agg", "p_nmds_agg", "b_recon"]),
    ("02 Product mapping", "Account maps, combined base, rate analysis, future-state map, dashboard and workbook.",
     ["m_pnfp_map", "m_pnfp_s", "m_snv_maps", "m_pnfp_base", "m_snv_base", "m_combined", "m_rate", "m_future", "m_dash", "o_xl_map", "o_html_map"]),
    ("03 WAR history", "pg_products -> balance-weighted WAR pivot, one row per product.", ["e_pg", "e_war", "f_betas"]),
    ("03 Fed funds and cycles", "Month-end fed funds and the five cycle windows.", ["p_ffr", "e_effr", "i_cycles", "e_cycles"]),
    ("03 Beta methods and save", "Endpoint, linear, cubic (UP_FFR and EFFR) x 5 cycles x 4 lags; save long and short results.",
     ["e_methods", "e_final", "e_short", "o_final_csv", "o_ccalc", "e_chk"]),
    ("04 Legacy inputs", "Legacy betas and truncations from CSV.", ["i_lb_snv", "i_lb_pnfp", "i_trunc", "g_snv_bv", "g_pnfp_bv", "g_trunc", "g_lb", "o_lb_tbl"]),
    ("04 Bank roll-ups", "PNFP via FTP account link, SNV via almAccountNum; join calculated betas; union.",
     ["p_ftp", "g_pnfp_beta", "g_pnfp_det", "g_lpnfp_betas", "g_snv_beta", "g_snv_det", "g_lsnv_betas", "g_consol", "o_consol_tbl", "g_chk"]),
    ("05 Configuration", "Load this version's tables, apply and validate product_config.csv.", ["i_pcfg", "f_config", "r_load", "r_pc"]),
    ("05 Workbooks and recommended beta", "Beta Detail, executive summary, PG workbook, summary tabs, beta_summary.",
     ["r_detail", "r_exec", "r_pgwb", "r_cfgdata", "r_save", "o_xl_beta", "o_xl_consol", "o_bs_csv", "o_bs_tbl"]),
    ("06 / 07 Presentation", "Fed funds chart, PG dashboard, account-to-product percentiles.",
     ["nb06", "nb07", "x_ffr", "x_pgdash", "x_a2p", "o_html_ffr", "o_html_pg", "o_xl_a2p"]),
    ("08 Version compare", "Inputs and beta_summary of two versions -> comparison workbook and cross-version view.", ["nb08", "f_compare", "x_compare", "o_cmp", "o_allv"]),
    ("09 Promotion (guarded)", "Dry run, typed confirmation, backup, then write to targets.", ["nb09", "i_promo", "f_promote", "z_dry", "z_exec", "t_prod", "t_ref", "o_promo_log"]),
]

# ---------------------------------------------------------------- integrity checks
ids = [n["id"] for n in NODES]
assert len(ids) == len(set(ids)), "duplicate node ids: %s" % [i for i in ids if ids.count(i) > 1]
idset = set(ids)
bad = [(x["s"], x["t"]) for x in EDGES if x["s"] not in idset or x["t"] not in idset]
assert not bad, "dangling edges: %s" % bad
linked = {x["s"] for x in EDGES} | {x["t"] for x in EDGES}
assert not (idset - linked), "orphan nodes: %s" % sorted(idset - linked)
missing = [i for _, _, ns in STAGES for i in ns if i not in idset]
assert not missing, "stage ids missing: %s" % missing
assert len({(x["s"], x["t"], x["type"]) for x in EDGES}) == len(EDGES), "duplicate edges"

from collections import Counter
print(f"{len(NODES)} nodes | {len(EDGES)} edges | {len(STAGES)} run-order stages | {sum(1 for n in NODES if n['warn'])} flagged issues")
for i, (k, nm) in enumerate(LAYERS):
    print(f"  L{i:<2} {nm:<36} {sum(1 for n in NODES if n['layer'] == i):>3} nodes")
print("edge types:", dict(Counter(x["type"] for x in EDGES)))

In [0]:
import json

LAYER_COLORS = ["#5c6c8f", "#7a5cc8", "#2f7fd6", "#b8811a", "#128f7c", "#3a9fbf",
                "#c8532f", "#d0457a", "#6b8e23", "#8a6d3b", "#4f6d7a"]

payload = {
    "layers": [{"id": k, "name": nm, "color": LAYER_COLORS[i % len(LAYER_COLORS)]} for i, (k, nm) in enumerate(LAYERS)],
    "nodes": NODES,
    "edges": EDGES,
    "stages": [{"title": t, "desc": d, "nodes": ns} for t, d, ns in STAGES],
}
data_json = json.dumps(payload).replace("</", "<\\/")

HTML = r"""
<meta charset="utf-8"/>
<style>
  :root{--bg:#f4f6fb;--bg2:#ffffff;--panel:#ffffffe6;--ink:#131a2a;--muted:#5b6579;--border:#dbe1ee;
        --accent:#2f7fd6;--gold:#b8811a;--coral:#c8532f;--edge:rgba(60,80,120,0.22);--band:rgba(60,80,120,0.045);}
  @media (prefers-color-scheme: dark){:root{--bg:#090d17;--bg2:#111a2c;--panel:#111a2ce6;--ink:#e8edf8;--muted:#8593ac;
        --border:#223049;--accent:#6fb6ff;--gold:#f0bc5e;--coral:#ff9a79;--edge:rgba(150,170,210,0.24);--band:rgba(150,170,210,0.05);}}
  *{box-sizing:border-box;} html,body{height:900px;margin:0;}
  body{background:var(--bg);color:var(--ink);font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",system-ui,sans-serif;overflow:hidden;}
  .mono{font-family:ui-monospace,"Cascadia Code",Menlo,Consolas,monospace;}
  #app{position:relative;width:100%;height:900px;display:flex;flex-direction:column;}
  header{display:flex;align-items:baseline;gap:18px;padding:12px 18px;border-bottom:1px solid var(--border);background:var(--panel);z-index:5;}
  header h1{font-size:16px;margin:0;font-weight:700;} header h1 span{font-weight:500;color:var(--muted);font-size:13px;margin-left:8px;}
  .stats{display:flex;gap:16px;margin-left:auto;font-size:12px;color:var(--muted);} .stats b{color:var(--ink);font-variant-numeric:tabular-nums;}
  #main{flex:1;position:relative;overflow:hidden;}
  #cv{position:absolute;inset:0;cursor:grab;} #cv.drag{cursor:grabbing;}
  .panel{position:absolute;background:var(--panel);border:1px solid var(--border);border-radius:10px;backdrop-filter:blur(6px);z-index:4;}
  #side{top:12px;left:12px;width:280px;max-height:calc(100% - 24px);display:flex;flex-direction:column;overflow:hidden;}
  .tabs{display:flex;border-bottom:1px solid var(--border);} .tab{flex:1;text-align:center;font-size:12px;padding:9px 0;cursor:pointer;color:var(--muted);user-select:none;}
  .tab.on{color:var(--accent);box-shadow:inset 0 -2px 0 var(--accent);font-weight:600;}
  .pane{padding:12px;overflow-y:auto;display:none;flex-direction:column;gap:12px;} .pane.on{display:flex;}
  .lbl{font-size:10px;text-transform:uppercase;letter-spacing:.08em;color:var(--muted);margin:0 0 6px;font-weight:600;}
  input[type=text]{width:100%;padding:7px 9px;border-radius:7px;border:1px solid var(--border);background:var(--bg2);color:var(--ink);font-size:12.5px;outline:none;}
  input[type=text]:focus{border-color:var(--accent);}
  .res{max-height:140px;overflow-y:auto;margin-top:5px;display:flex;flex-direction:column;gap:1px;}
  .item{font-size:12px;padding:4px 7px;border-radius:5px;cursor:pointer;display:flex;align-items:center;gap:6px;}
  .item:hover{background:var(--border);} .sw{width:9px;height:9px;border-radius:3px;flex:0 0 auto;}
  .seg{display:flex;border:1px solid var(--border);border-radius:7px;overflow:hidden;} .seg div{flex:1;text-align:center;font-size:12px;padding:5px 0;cursor:pointer;}
  .seg div.on{background:var(--accent);color:#fff;}
  .row{display:flex;align-items:center;gap:8px;font-size:12.5px;padding:2px 0;cursor:pointer;user-select:none;}
  .row.off{opacity:.35;} .row .n{flex:1;} .row .c{color:var(--muted);font-size:11px;font-variant-numeric:tabular-nums;}
  .chk{display:flex;align-items:center;gap:7px;font-size:12px;cursor:pointer;user-select:none;padding:2px 0;}
  .key{display:inline-block;width:22px;height:0;border-top:2px solid var(--muted);vertical-align:middle;}
  .stage{border:1px solid var(--border);border-radius:8px;padding:8px 9px;cursor:pointer;background:var(--bg2);}
  .stage:hover,.stage.on{border-color:var(--accent);} .stage .t{font-size:12.5px;font-weight:600;} .stage .d{font-size:11.5px;color:var(--muted);margin-top:3px;line-height:1.35;}
  .stage .num{display:inline-block;min-width:20px;color:var(--accent);font-variant-numeric:tabular-nums;}
  .iss{border-left:3px solid var(--coral);padding:6px 8px;background:var(--bg2);border-radius:0 7px 7px 0;cursor:pointer;}
  .iss .t{font-size:12px;font-weight:600;} .iss .d{font-size:11.5px;color:var(--muted);margin-top:2px;line-height:1.35;}
  #det{top:12px;right:12px;width:340px;max-height:calc(100% - 64px);overflow-y:auto;padding:14px;display:none;flex-direction:column;gap:9px;}
  #det.on{display:flex;} #det .nm{font-size:13.5px;font-weight:700;word-break:break-all;padding-right:20px;}
  #det .fq{font-size:11px;color:var(--muted);word-break:break-all;} #det .x{position:absolute;top:8px;right:10px;cursor:pointer;color:var(--muted);font-size:16px;}
  .tags{display:flex;gap:5px;flex-wrap:wrap;} .tag{font-size:10px;padding:2px 7px;border-radius:99px;border:1px solid var(--border);color:var(--muted);text-transform:uppercase;letter-spacing:.04em;}
  #det .desc{font-size:12.5px;line-height:1.45;} #det ol{margin:0;padding-left:18px;font-size:12px;line-height:1.45;}
  #det table{border-collapse:collapse;font-size:11.5px;width:100%;} #det td{border-top:1px solid var(--border);padding:3px 4px;vertical-align:top;} #det td:first-child{color:var(--muted);width:42%;}
  .warn{border:1px solid var(--coral);color:var(--ink);background:rgba(200,83,47,.08);border-radius:7px;padding:7px 9px;font-size:12px;line-height:1.4;}
  .lst{display:flex;flex-direction:column;max-height:120px;overflow-y:auto;gap:1px;}
  #hint{bottom:12px;left:304px;font-size:11px;color:var(--muted);padding:6px 10px;}
  #zoom{position:absolute;bottom:12px;right:12px;display:flex;gap:6px;z-index:4;}
  #zoom button{width:30px;height:30px;border-radius:8px;border:1px solid var(--border);background:var(--panel);color:var(--ink);font-size:15px;cursor:pointer;}
  #zoom button:hover{border-color:var(--accent);color:var(--accent);}
  #tip{position:absolute;pointer-events:none;background:var(--bg2);border:1px solid var(--border);border-radius:6px;padding:5px 9px;font-size:12px;box-shadow:0 4px 14px rgba(0,0,0,.18);display:none;z-index:6;white-space:nowrap;}
  ::-webkit-scrollbar{width:7px;} ::-webkit-scrollbar-thumb{background:var(--border);border-radius:99px;}
</style>
<div id="app">
  <header><h1>deposit study 2027v16<span>- deposit beta study: lineage, notebooks &amp; model logic</span></h1>
    <div class="stats"><span><b id="sN">0</b> nodes</span><span><b id="sE">0</b> edges</span><span><b id="sL">0</b> layers</span>
    <span><b id="sS">0</b> production sources</span><span><b id="sI">0</b> flagged issues</span></div></header>
  <div id="main">
    <canvas id="cv"></canvas>
    <div id="side" class="panel">
      <div class="tabs"><div class="tab on" data-p="pExp">Explore</div><div class="tab" data-p="pWf">Run order</div><div class="tab" data-p="pIss">Issues</div></div>
      <div class="pane on" id="pExp">
        <div><p class="lbl">Search</p><input id="q" class="mono" type="text" placeholder="e.g. nmds_agg, lpnfp_betas, product_config" autocomplete="off"/><div id="qr" class="res"></div></div>
        <div><p class="lbl">Legacy bank</p><div class="seg" id="ent"><div class="on" data-v="All">All</div><div data-v="SNV">SNV</div><div data-v="PNFP">PNFP</div></div></div>
        <div><p class="lbl">Edge types</p>
          <label class="chk"><input type="checkbox" data-t="d" checked/><span class="key" style="border-top:2px solid var(--muted)"></span>Data flow</label>
          <label class="chk"><input type="checkbox" data-t="o" checked/><span class="key" style="border-top:2px solid var(--accent)"></span>Run order / call</label>
          <label class="chk"><input type="checkbox" data-t="p" checked/><span class="key" style="border-top:2px dashed var(--gold)"></span>Parameter / config injection</label>
          <label class="chk"><input type="checkbox" data-t="r" checked/><span class="key" style="border-top:2px dotted var(--coral)"></span>Recon / check</label>
        </div>
        <div><p class="lbl">Layers (click to toggle)</p><div id="lg"></div></div>
        <div><p class="lbl">Node kinds</p><div id="kd" style="font-size:12px;line-height:1.7"></div></div>
      </div>
      <div class="pane" id="pWf"><p class="lbl">Run order - click a stage to spotlight its nodes</p><div id="wf" style="display:flex;flex-direction:column;gap:6px"></div></div>
      <div class="pane" id="pIss"><p class="lbl">Findings from the 2027v16 review</p><div id="is" style="display:flex;flex-direction:column;gap:6px"></div></div>
    </div>
    <div id="det" class="panel"><span class="x" id="dx">&times;</span>
      <div class="nm mono" id="dN"></div><div class="fq mono" id="dF"></div><div class="tags" id="dT"></div>
      <div class="desc" id="dD"></div><div id="dW"></div><div id="dL"></div><div id="dP"></div><div class="fq mono" id="dS"></div>
      <div><p class="lbl">Upstream (<span id="dUc"></span>)</p><div class="lst mono" id="dU"></div></div>
      <div><p class="lbl">Downstream (<span id="dDc"></span>)</p><div class="lst mono" id="dDn"></div></div>
      <div class="chk"><input type="checkbox" id="deep" checked/>Highlight full transitive lineage</div>
    </div>
    <div id="hint" class="panel">scroll to zoom · drag to pan · drag a node to move it · click for details · Esc to clear</div>
    <div id="zoom"><button id="zo" title="Zoom out">-</button><button id="zf" title="Fit">&#10530;</button><button id="zi" title="Zoom in">+</button></div>
    <div id="tip"></div>
  </div>
</div>
<script>
(function(){
const D = __DATA__;
const LAY = D.layers, NODES = D.nodes, EDGES = D.edges, STAGES = D.stages;
const KIND = {table:"\u25A6", view:"\u25C7", function:"\u0192", notebook:"\u25A4", config:"\u2699", file:"\u25AD", query:"\u2315", external:"\u2601"};
const KNAME = {table:"UC table (production or sandbox)", view:"Temp view / DataFrame", function:"Python function / step", notebook:"Study notebook", config:"Config file (inputs/)", file:"CSV / output file", query:"SQL check / query", external:"Promotion target"};
const $ = id => document.getElementById(id);
const byId = {}; NODES.forEach(n => { byId[n.id] = n; n.up = []; n.dn = []; });
EDGES.forEach(e => { byId[e.s].dn.push(e); byId[e.t].up.push(e); });
const css = () => { const s = getComputedStyle(document.documentElement); const g = k => s.getPropertyValue(k).trim();
  return {bg2:g("--bg2"), ink:g("--ink"), muted:g("--muted"), border:g("--border"), accent:g("--accent"), gold:g("--gold"), coral:g("--coral"), edge:g("--edge"), band:g("--band")}; };
let C = css();

// ---------- layout: layered columns + barycenter ordering
const COLW = 260, NW = 200, NH = 30, VG = 14;
const cols = LAY.map(() => []); NODES.forEach(n => cols[n.layer].push(n));
function assignY(){ cols.forEach(col => { const h = col.length * (NH + VG); col.forEach((n, i) => { n.y = i * (NH + VG) - h / 2; }); }); }
const avg = a => a.reduce((x, y) => x + y, 0) / a.length;
assignY();
for (let it = 0; it < 8; it++) {
  for (let c = 1; c < cols.length; c++) { cols[c].forEach(n => { const ys = n.up.map(e => byId[e.s]).filter(m => m.layer < n.layer).map(m => m.y); n._b = ys.length ? avg(ys) : n.y; }); cols[c].sort((a, b) => a._b - b._b); assignY(); }
  for (let c = cols.length - 2; c >= 0; c--) { cols[c].forEach(n => { const ys = n.dn.map(e => byId[e.t]).filter(m => m.layer > n.layer).map(m => m.y); n._b = ys.length ? avg(ys) : n.y; }); cols[c].sort((a, b) => a._b - b._b); assignY(); }
}
NODES.forEach(n => { n.x = n.layer * COLW; });

// ---------- state
const layerOn = LAY.map(() => true), edgeOn = {d:true, o:true, p:true, r:true};
let entity = "All", sel = null, spot = null, hover = null, deep = true;
let scale = 0.5, tx = 0, ty = 0;
const cv = $("cv"), ctx = cv.getContext("2d"), main = $("main");
const entOk = n => entity === "All" || n.entity === entity || n.entity === "Both";
const vis = n => layerOn[n.layer] && entOk(n);
const eVis = e => edgeOn[e.type] && vis(byId[e.s]) && vis(byId[e.t]);

function lineage(id){
  const up = new Set(), dn = new Set();
  const walk = (start, dir, set) => { const st = [start]; while (st.length) { const cur = st.pop();
    (dir === "up" ? byId[cur].up : byId[cur].dn).forEach(e => { if (!eVis(e)) return; const nx = dir === "up" ? e.s : e.t;
      if (!set.has(nx) && nx !== start) { set.add(nx); if (deep) st.push(nx); } }); } };
  walk(id, "up", up); walk(id, "dn", dn); return {up, dn};
}
let LIN = null;
function focusSet(){ if (sel) { const s = new Set([sel, ...LIN.up, ...LIN.dn]); return s; } if (spot) return new Set(spot); return null; }

// ---------- drawing
function resize(){ const r = main.getBoundingClientRect(), dpr = window.devicePixelRatio || 1;
  cv.width = r.width * dpr; cv.height = r.height * dpr; cv.style.width = r.width + "px"; cv.style.height = r.height + "px"; draw(); }
function rr(x, y, w, h, r){ ctx.beginPath(); ctx.moveTo(x + r, y); ctx.arcTo(x + w, y, x + w, y + h, r); ctx.arcTo(x + w, y + h, x, y + h, r); ctx.arcTo(x, y + h, x, y, r); ctx.arcTo(x, y, x + w, y, r); ctx.closePath(); }
function bounds(){ const vs = NODES.filter(vis); if (!vs.length) return {x0:0, y0:0, x1:1, y1:1};
  return {x0: Math.min(...vs.map(n => n.x)) - 30, x1: Math.max(...vs.map(n => n.x)) + NW + 30, y0: Math.min(...vs.map(n => n.y)) - 70, y1: Math.max(...vs.map(n => n.y)) + NH + 30}; }
function draw(){
  const dpr = window.devicePixelRatio || 1;
  ctx.setTransform(1, 0, 0, 1, 0, 0); ctx.clearRect(0, 0, cv.width, cv.height);
  ctx.setTransform(dpr * scale, 0, 0, dpr * scale, dpr * tx, dpr * ty);
  const F = focusSet(), b = bounds();
  // layer bands
  LAY.forEach((L, i) => { if (!layerOn[i]) return; const x = i * COLW - 22;
    ctx.fillStyle = C.band; rr(x, b.y0, NW + 44, b.y1 - b.y0, 12); ctx.fill();
    ctx.fillStyle = L.color; ctx.fillRect(x + 12, b.y0 + 14, 18, 4);
    ctx.fillStyle = C.ink; ctx.font = "600 13px -apple-system,Segoe UI,sans-serif"; ctx.textBaseline = "alphabetic";
    ctx.fillText((i) + ". " + L.name, x + 12, b.y0 + 36);
    ctx.fillStyle = C.muted; ctx.font = "11px -apple-system,Segoe UI,sans-serif"; ctx.fillText(cols[i].filter(vis).length + " nodes", x + 12, b.y0 + 52); });
  // edges
  const hiE = e => F && F.has(e.s) && F.has(e.t) && (!sel || (e.s === sel || e.t === sel || (LIN.up.has(e.s) && (LIN.up.has(e.t) || e.t === sel)) || (LIN.dn.has(e.t) && (LIN.dn.has(e.s) || e.s === sel))));
  const drawEdge = (e, hi) => { const s = byId[e.s], t = byId[e.t];
    const x1 = s.x + NW, y1 = s.y + NH / 2, x2 = t.x, y2 = t.y + NH / 2; const back = t.layer <= s.layer;
    const dx = back ? 120 : Math.max(50, (x2 - x1) / 2);
    ctx.beginPath(); ctx.moveTo(x1, y1);
    if (back) { const lift = Math.min(y1, y2) - 40 - Math.abs(x1 - x2) * 0.08; ctx.bezierCurveTo(x1 + dx, lift, x2 - dx, lift, x2, y2); }
    else ctx.bezierCurveTo(x1 + dx, y1, x2 - dx, y2, x2, y2);
    const col = e.type === "p" ? C.gold : e.type === "r" ? C.coral : e.type === "o" ? C.accent : C.muted;
    ctx.setLineDash(e.type === "p" ? [6, 4] : e.type === "r" ? [2, 4] : []);
    ctx.strokeStyle = hi ? col : (F ? C.edge : (e.type === "d" ? C.edge : col)); ctx.globalAlpha = hi ? 0.95 : (F ? 0.25 : (e.type === "d" ? 1 : 0.45));
    ctx.lineWidth = hi ? 2.1 : 1.2; ctx.stroke(); ctx.setLineDash([]);
    ctx.beginPath(); ctx.moveTo(x2, y2); ctx.lineTo(x2 - 7, y2 - 3.5); ctx.lineTo(x2 - 7, y2 + 3.5); ctx.closePath(); ctx.fillStyle = ctx.strokeStyle; ctx.fill();
    if (hi && e.label && scale > 0.55) { ctx.globalAlpha = 1; ctx.font = "10px ui-monospace,Menlo,monospace"; const mx = (x1 + x2) / 2, my = back ? Math.min(y1, y2) - 34 : (y1 + y2) / 2 - 4;
      const w = ctx.measureText(e.label).width + 8; ctx.fillStyle = C.bg2; ctx.fillRect(mx - w / 2, my - 10, w, 13); ctx.fillStyle = col; ctx.textAlign = "center"; ctx.fillText(e.label, mx, my); ctx.textAlign = "left"; }
    ctx.globalAlpha = 1; };
  const vE = EDGES.filter(eVis); vE.filter(e => !hiE(e)).forEach(e => drawEdge(e, false)); vE.filter(hiE).forEach(e => drawEdge(e, true));
  // nodes
  NODES.forEach(n => { if (!vis(n)) return; const dim = F && !F.has(n.id); ctx.globalAlpha = dim ? 0.18 : 1;
    const L = LAY[n.layer]; rr(n.x, n.y, NW, NH, 7); ctx.fillStyle = C.bg2; ctx.fill();
    ctx.lineWidth = (n.id === sel) ? 2.4 : (n.id === hover ? 1.8 : 1); ctx.strokeStyle = (n.id === sel || n.id === hover) ? C.accent : C.border; ctx.stroke();
    ctx.fillStyle = L.color; rr(n.x, n.y, 5, NH, 2); ctx.fill();
    ctx.fillStyle = L.color; ctx.font = "13px ui-monospace,Menlo,monospace"; ctx.textBaseline = "middle"; ctx.fillText(KIND[n.kind] || "?", n.x + 11, n.y + NH / 2 + 1);
    ctx.fillStyle = C.ink; ctx.font = (n.kind === "table" || n.kind === "view" ? "600 " : "") + "11.5px ui-monospace,Menlo,monospace";
    let t = n.label, maxW = NW - 44; while (ctx.measureText(t).width > maxW && t.length > 4) t = t.slice(0, -2); if (t !== n.label) t = t.slice(0, -1) + "\u2026";
    ctx.fillText(t, n.x + 28, n.y + NH / 2 + 1);
    if (n.entity !== "Both") { ctx.font = "9px -apple-system,Segoe UI,sans-serif"; ctx.fillStyle = C.muted; ctx.textAlign = "right"; ctx.fillText(n.entity, n.x + NW - (n.warn ? 16 : 6), n.y + NH / 2 + 1); ctx.textAlign = "left"; }
    if (n.warn) { ctx.beginPath(); ctx.arc(n.x + NW - 7, n.y + 7, 4, 0, Math.PI * 2); ctx.fillStyle = C.coral; ctx.fill(); }
    ctx.globalAlpha = 1; ctx.textBaseline = "alphabetic"; });
}

// ---------- view helpers
function fit(){ const r = main.getBoundingClientRect(), b = bounds(); const padL = 300, padR = sel ? 360 : 20;
  const w = r.width - padL - padR, h = r.height - 40; scale = Math.max(0.15, Math.min(1.4, Math.min(w / (b.x1 - b.x0), h / (b.y1 - b.y0))));
  tx = padL + (w - (b.x1 - b.x0) * scale) / 2 - b.x0 * scale; ty = 20 + (h - (b.y1 - b.y0) * scale) / 2 - b.y0 * scale; draw(); }
function centerOn(n){ const r = main.getBoundingClientRect(); if (scale < 0.7) scale = 0.85;
  tx = (r.width - 340 + 290) / 2 - (n.x + NW / 2) * scale; ty = r.height / 2 - (n.y + NH / 2) * scale; draw(); }
function zoomAt(f, cx, cy){ const ns = Math.max(0.12, Math.min(3, scale * f)); tx = cx - (cx - tx) * ns / scale; ty = cy - (cy - ty) * ns / scale; scale = ns; draw(); }
const toWorld = (mx, my) => ({x: (mx - tx) / scale, y: (my - ty) / scale});
function hit(mx, my){ const p = toWorld(mx, my); for (let i = NODES.length - 1; i >= 0; i--) { const n = NODES[i]; if (vis(n) && p.x >= n.x && p.x <= n.x + NW && p.y >= n.y && p.y <= n.y + NH) return n; } return null; }

// ---------- selection + detail panel
const esc = s => String(s).replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
function select(id, center){
  sel = id; spot = null; document.querySelectorAll(".stage").forEach(s => s.classList.remove("on"));
  if (!id) { LIN = null; $("det").classList.remove("on"); draw(); return; }
  const n = byId[id]; if (!vis(n)) { layerOn[n.layer] = true; if (!entOk(n)) setEntity("All"); buildLegend(); }
  LIN = lineage(id); const L = LAY[n.layer];
  $("dN").textContent = n.label; $("dF").textContent = n.fq !== n.label ? n.fq : "";
  $("dT").innerHTML = `<span class="tag" style="border-color:${L.color};color:${L.color}">L${n.layer} ${esc(L.name)}</span><span class="tag">${esc(n.kind)}</span><span class="tag">${esc(n.entity === "Both" ? "SNV + PNFP" : n.entity)}</span>`;
  $("dD").textContent = n.desc;
  $("dW").innerHTML = n.warn ? `<div class="warn"><b>Review:</b> ${esc(n.warn)}</div>` : "";
  $("dL").innerHTML = n.logic.length ? `<p class="lbl">Logic</p><ol>${n.logic.map(s => `<li>${esc(s)}</li>`).join("")}</ol>` : "";
  const pk = Object.keys(n.params); $("dP").innerHTML = pk.length ? `<p class="lbl">Parameters / rules</p><table>${pk.map(k => `<tr><td class="mono">${esc(k)}</td><td>${esc(n.params[k])}</td></tr>`).join("")}</table>` : "";
  $("dS").textContent = n.src ? "source: " + n.src : "";
  const li = (list, key) => list.map(e => { const m = byId[e[key]]; return `<div class="item" data-id="${m.id}"><span class="sw" style="background:${LAY[m.layer].color}"></span>${esc(m.label)}${e.label ? ` <span style="color:var(--muted)">(${esc(e.label)})</span>` : ""}</div>`; }).join("");
  $("dU").innerHTML = li(n.up, "s"); $("dUc").textContent = n.up.length; $("dDn").innerHTML = li(n.dn, "t"); $("dDc").textContent = n.dn.length;
  $("det").classList.add("on"); if (center) centerOn(n); else draw();
}
$("det").addEventListener("click", ev => { const it = ev.target.closest(".item"); if (it) select(it.dataset.id, true); });
$("dx").onclick = () => select(null);
$("deep").onchange = ev => { deep = ev.target.checked; if (sel) { LIN = lineage(sel); draw(); } };

// ---------- sidebar
document.querySelectorAll(".tab").forEach(t => t.onclick = () => { document.querySelectorAll(".tab").forEach(x => x.classList.remove("on")); document.querySelectorAll(".pane").forEach(x => x.classList.remove("on")); t.classList.add("on"); $(t.dataset.p).classList.add("on"); });
function buildLegend(){ $("lg").innerHTML = LAY.map((L, i) => `<div class="row ${layerOn[i] ? "" : "off"}" data-i="${i}"><span class="sw" style="background:${L.color}"></span><span class="n">${i}. ${esc(L.name)}</span><span class="c">${cols[i].length}</span></div>`).join(""); }
$("lg").onclick = ev => { const r = ev.target.closest(".row"); if (!r) return; const i = +r.dataset.i; layerOn[i] = !layerOn[i]; buildLegend(); if (sel && !vis(byId[sel])) select(null); else if (sel) LIN = lineage(sel); draw(); };
buildLegend();
$("kd").innerHTML = Object.keys(KIND).map(k => `<div><span class="mono" style="display:inline-block;width:18px;text-align:center">${KIND[k]}</span> ${KNAME[k]}</div>`).join("") + `<div><span style="display:inline-block;width:18px;text-align:center;color:var(--coral)">&#9679;</span> Flagged for review</div>`;
function setEntity(v){ entity = v; document.querySelectorAll("#ent div").forEach(d => d.classList.toggle("on", d.dataset.v === v)); }
$("ent").onclick = ev => { const d = ev.target.closest("div[data-v]"); if (!d) return; setEntity(d.dataset.v); if (sel && !vis(byId[sel])) select(null); else if (sel) LIN = lineage(sel); fit(); };
document.querySelectorAll("input[data-t]").forEach(c => c.onchange = () => { edgeOn[c.dataset.t] = c.checked; if (sel) LIN = lineage(sel); draw(); });
$("q").oninput = () => { const q = $("q").value.trim().toLowerCase(); if (!q) { $("qr").innerHTML = ""; return; }
  const hits = NODES.filter(n => (n.label + " " + n.fq + " " + n.desc).toLowerCase().includes(q)).slice(0, 30);
  $("qr").innerHTML = hits.length ? hits.map(n => `<div class="item mono" data-id="${n.id}"><span class="sw" style="background:${LAY[n.layer].color}"></span>${esc(n.label)}</div>`).join("") : `<div class="item" style="color:var(--muted)">no matches</div>`; };
$("qr").onclick = ev => { const it = ev.target.closest(".item[data-id]"); if (it) select(it.dataset.id, true); };
$("wf").innerHTML = STAGES.map((s, i) => `<div class="stage" data-i="${i}"><div class="t"><span class="num">${i + 1}</span>${esc(s.title)}</div><div class="d">${esc(s.desc)}</div></div>`).join("");
$("wf").onclick = ev => { const st = ev.target.closest(".stage"); if (!st) return; const s = STAGES[+st.dataset.i];
  sel = null; LIN = null; $("det").classList.remove("on"); spot = s.nodes; document.querySelectorAll(".stage").forEach(x => x.classList.toggle("on", x === st));
  s.nodes.forEach(id => { layerOn[byId[id].layer] = true; }); setEntity("All"); buildLegend();
  const ns = s.nodes.map(id => byId[id]); const cx = avg(ns.map(n => n.x + NW / 2)), cy = avg(ns.map(n => n.y + NH / 2)); const r = main.getBoundingClientRect();
  scale = Math.max(scale, 0.6); tx = (r.width + 290) / 2 - cx * scale; ty = r.height / 2 - cy * scale; draw(); };
const ISS = NODES.filter(n => n.warn);
$("is").innerHTML = ISS.map(n => `<div class="iss" data-id="${n.id}"><div class="t mono">${esc(n.label)}</div><div class="d">${esc(n.warn)}</div></div>`).join("");
$("is").onclick = ev => { const it = ev.target.closest(".iss"); if (it) select(it.dataset.id, true); };

// ---------- canvas interaction
let down = null, moved = false;
cv.addEventListener("mousedown", ev => { const r = cv.getBoundingClientRect(), mx = ev.clientX - r.left, my = ev.clientY - r.top; const n = hit(mx, my);
  down = {mx, my, tx, ty, n, nx: n ? n.x : 0, ny: n ? n.y : 0}; moved = false; if (!n) cv.classList.add("drag"); });
window.addEventListener("mousemove", ev => { const r = cv.getBoundingClientRect(), mx = ev.clientX - r.left, my = ev.clientY - r.top;
  if (down) { const dx = mx - down.mx, dy = my - down.my; if (Math.abs(dx) + Math.abs(dy) > 3) moved = true;
    if (down.n) { down.n.x = down.nx + dx / scale; down.n.y = down.ny + dy / scale; } else { tx = down.tx + dx; ty = down.ty + dy; } draw(); return; }
  const n = (mx >= 0 && my >= 0 && mx <= r.width && my <= r.height) ? hit(mx, my) : null; const tip = $("tip");
  if ((n ? n.id : null) !== hover) { hover = n ? n.id : null; draw(); }
  if (n) { tip.style.display = "block"; tip.innerHTML = `<b class="mono">${esc(n.label)}</b> <span style="color:var(--muted)">${esc(KNAME[n.kind])} · ${esc(LAY[n.layer].name)}</span>`; tip.style.left = (mx + 14) + "px"; tip.style.top = (my + 12) + "px"; cv.style.cursor = "pointer"; }
  else { tip.style.display = "none"; cv.style.cursor = ""; } });
window.addEventListener("mouseup", () => { if (down && !moved) { if (down.n) select(down.n.id === sel ? null : down.n.id, false); else select(null); } down = null; cv.classList.remove("drag"); });
cv.addEventListener("wheel", ev => { ev.preventDefault(); const r = cv.getBoundingClientRect(); zoomAt(ev.deltaY < 0 ? 1.12 : 1 / 1.12, ev.clientX - r.left, ev.clientY - r.top); }, {passive: false});
window.addEventListener("keydown", ev => { if (ev.key === "Escape") { spot = null; document.querySelectorAll(".stage").forEach(s => s.classList.remove("on")); select(null); } });
$("zi").onclick = () => { const r = main.getBoundingClientRect(); zoomAt(1.2, r.width / 2, r.height / 2); };
$("zo").onclick = () => { const r = main.getBoundingClientRect(); zoomAt(1 / 1.2, r.width / 2, r.height / 2); };
$("zf").onclick = fit;
window.matchMedia("(prefers-color-scheme: dark)").addEventListener("change", () => { C = css(); draw(); });
window.addEventListener("resize", resize);

$("sN").textContent = NODES.length; $("sE").textContent = EDGES.length; $("sL").textContent = LAY.length;
$("sS").textContent = cols[0].length; $("sI").textContent = ISS.length;
resize(); fit();
})();
</script>
"""

displayHTML(HTML.replace("__DATA__", data_json))